# VAD Adapter

> The typed voice-activity-detection task contract — the `VADAdapter` ABC + the `VADToolProtocol` structural contract (capability-unit Option C, pass-2 Thread 3).

In [ ]:
#| default_exp adapter

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from abc import abstractmethod
from pathlib import Path
from typing import Any, ClassVar, Dict, Protocol, Union, runtime_checkable

from cjm_substrate.core.adapter import TaskAdapter

# Canonical home (stage 8 / PILLAR 1c): the data noun lives in
# cjm-capability-primitives, so the pure-compute tool depends on it, never on
# this adapter machinery.
from cjm_capability_primitives.vad import VADResult

## Tool protocol (born-final — stage 8)

`required_tool_protocol` is the REAL structural contract derived from the native VAD tool surface: a VAD tool capability exposes a pure-compute `detect_speech(audio, **kwargs) -> VADResult` plus `get_current_config()` (the adapter hashes the effective config for its cache key). `db_path` is deliberately NOT on the protocol — the adapter resolves storage from the substrate-injected `PLUGIN_DATA_DIR`, so persistence stays off the pure-compute tool.

Surface-based compatibility (stage 4) gates auto-binding: only tools whose recorded structural surface carries `detect_speech` + `get_current_config` match. **Input contract:** like the transcription adapter, the caller guarantees MODEL-READY audio — resampling / mono / decode happens upstream (ffmpeg `convert`, soxr), never in the tool (silero-vad's librosa load+resample retired on migration; the tool reads the ready audio via soundfile).

In [ ]:
#| export
@runtime_checkable
class VADToolProtocol(Protocol):
    """Structural contract for voice-activity-detection tool capabilities
    (born-final at stage 8 — derived from the native tool surface).

    Pure compute: `detect_speech` reads the model-ready audio + runs inference +
    builds the typed result. `get_current_config` supplies the effective config
    the generic adapter hashes for its cache key. Persistence is NOT here — the
    adapter owns it (the native-surface seam)."""
    def detect_speech(self, audio: Union[str, Path], **kwargs) -> VADResult: ...
    def get_current_config(self) -> Dict[str, Any]: ...

In [ ]:
#| export
class VADAdapter(TaskAdapter):
    """Typed voice-activity-detection task adapter: model-ready audio in,
    `VADResult` out.

    Input contract: the caller guarantees MODEL-READY audio — format /
    sample-rate / channel handling happens upstream (ffmpeg `convert`), never
    in the adapter or tool.

    Native-surface model (stage 8 / PILLAR 1c): the TOOL is pure compute; the
    ADAPTER owns the cache + persistence bookends (see `GenericVADAdapter`) +
    the per-call `force` control. Storage resolves from the substrate-injected
    `PLUGIN_DATA_DIR`; `db_path` is not on the tool protocol.

    Implementations run in-worker beside their tool capability and are
    constructed with the bound tool instance: `AdapterClass(tool)` (mirrors
    `GraphStorageAdapter`). The result DTO is wire-registered ("vad.result"):
    returned values cross the worker boundary typed.
    """

    task_name: ClassVar[str] = "vad"
    required_tool_protocol: ClassVar[type] = VADToolProtocol

    def __init__(
        self,
        tool: VADToolProtocol,  # The bound tool capability instance (worker-side binding)
    ):
        self.tool = tool

    @abstractmethod
    def detect_speech(
        self,
        audio: Union[str, Path],  # Path to MODEL-READY audio (converted upstream)
        **kwargs,                 # Provenance + tool options
    ) -> VADResult:               # Typed VAD output
        """Detect speech segments in model-ready audio."""
        ...

In [ ]:
# Shape tests: abstract gate + ClassVars + __init__(tool) + a concrete impl
# returning the typed DTO + the protocol structurally matching a migrated tool
# surface (detect_speech + get_current_config), and NOT a fused-era execute tool.
from cjm_capability_primitives.vad import TimeRange

class _FakeTool:  # structural match, no inheritance
    def detect_speech(self, audio, **kwargs) -> VADResult:
        return VADResult(ranges=[TimeRange(start=0.0, end=1.0)], metadata={"src": audio})
    def get_current_config(self): return {"threshold": 0.5}


class _FakeVADAdapter(VADAdapter):
    def detect_speech(self, audio, **kwargs) -> VADResult:
        return self.tool.detect_speech(audio, **kwargs)


try:
    VADAdapter(_FakeTool())  # type: ignore[abstract]
    raise AssertionError("abstract ABC must not instantiate")
except TypeError:
    pass

impl = _FakeVADAdapter(_FakeTool())
out = impl.detect_speech("a.wav")
assert isinstance(out, VADResult) and out.ranges[0].end == 1.0
assert VADAdapter.task_name == "vad"
assert VADAdapter.required_tool_protocol is VADToolProtocol
assert isinstance(_FakeTool(), VADToolProtocol)

# the fused-era execute-shaped tool does NOT satisfy the real protocol
class _FusedEraTool:
    def execute(self, media_path, **kwargs): return {"ranges": []}
assert not isinstance(_FusedEraTool(), VADToolProtocol)
print("VADAdapter shape tests OK")

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()